# 2. Value at risk and expected shortfall

**Level:** L1. **Tags:** `risk`, `var`, `expected-shortfall`.
**Library:** `eq_var` (`python/equity/03-var-es-engine`).

**Plain idea.** VaR at 99% is the loss you exceed on one day in a hundred.
Expected shortfall (ES) is the average loss on those bad days, so it sees
how deep the tail goes.

**Formal statement** (McNeil, Frey and Embrechts, *Quantitative Risk
Management*, 2nd ed., sec. 2.3): for P&L $X$ and tail probability $\alpha$,
$\mathrm{VaR}_\alpha = -q_\alpha(X)$ and
$\mathrm{ES}_\alpha = -\frac{1}{\alpha}\int_0^\alpha q_u(X)\,du$.
For normal P&L with zero mean, $\mathrm{VaR} = -z_\alpha\sigma$ and
$\mathrm{ES} = \sigma\varphi(z_\alpha)/\alpha$.

In [ ]:
import _paths  # noqa: F401  (puts the library on sys.path)

In [ ]:
import math

import numpy as np
from eq_var.expected_shortfall import expected_shortfall, normal_es
from eq_var.historical_var import historical_var
from eq_var.parametric_var import parametric_var
from scipy.stats import norm

rng = np.random.default_rng(20261103)  # fixed seed: the notebook is deterministic

## Task 1: parametric VaR of a two-asset book

Exposures of 1,000,000 and -400,000 in two assets with daily vols 1.2% and
0.9% and correlation 0.6. Build the covariance `cov` and compute the 99%
one-day normal VaR with `parametric_var`. Store it as `var_p`.

In [ ]:
w = np.array([1_000_000.0, -400_000.0])
vol = np.array([0.012, 0.009])
corr = np.array([[1.0, 0.6], [0.6, 1.0]])
cov = np.outer(vol, vol) * corr
var_p = parametric_var(w, cov, alpha=0.01)
print(f"99% 1-day parametric VaR: {var_p:,.0f}")

In [ ]:
sigma_p = math.sqrt(w @ cov @ w)
assert abs(var_p - (-norm.ppf(0.01)) * sigma_p) < 1e-6

## Task 2: historical VaR and ES on simulated P&L

Draw 250,000 days of P&L from the same normal model (seeded above), then
compute historical VaR and ES at 99%. With this many days both should sit
close to the closed forms; with 250 days (one year) they would not.

In [ ]:
pnl = rng.multivariate_normal(np.zeros(2), cov, size=250_000) @ w
var_h = historical_var(pnl, alpha=0.01)
es_h = expected_shortfall(pnl, alpha=0.01)
es_n = normal_es(sigma_p, alpha=0.01)
print(f"historical VaR {var_h:,.0f}  ES {es_h:,.0f}  normal ES {es_n:,.0f}")

In [ ]:
z = norm.ppf(0.01)
assert abs(es_n - sigma_p * norm.pdf(z) / 0.01) < 1e-6
assert abs(var_h / var_p - 1) < 0.02
assert abs(es_h / es_n - 1) < 0.02
assert es_h > var_h

## Task 3: why one year of data is noisy

Repeat the historical VaR on 200 independent 250-day samples and measure the
spread. Store the relative standard deviation in `rel_sd`.

In [ ]:
samples = rng.multivariate_normal(np.zeros(2), cov, size=(200, 250)) @ w
vars_250 = np.array([historical_var(s, alpha=0.01) for s in samples])
rel_sd = vars_250.std() / var_p
print(f"relative s.d. of 250-day historical VaR: {rel_sd:.1%}")

In [ ]:
# Asymptotic s.e. of a sample quantile: sqrt(a(1-a)/n) / f(q), here relative to VaR.
asym = math.sqrt(0.01 * 0.99 / 250) / norm.pdf(z) / abs(z)
assert 0.5 * asym < rel_sd < 2.0 * asym

**Takeaways.** A one-year 99% VaR has a sampling error of roughly 10%
(the asymptotic variance of a sample quantile, Serfling, *Approximation
Theorems of Mathematical Statistics*, 1980, sec. 2.3). ES is always at least VaR.